# Lecture 5: Python Data Science Best Practices

How to "read" this lecture notebook
<details>
<summary>click to expand</summary>

As you go through this notebook (or any notebook for this class), you will encounter new concepts and python code that implements them -- just like you would see in a textbook. Of course, in a textbook, it's easy to read code and an explanation of what it does and think that you understand it.
<br />

### Learn by doing
But this notebook is different from a textbook because it allows you to not just read the code, but play with it. **You can and should try out changing the code that you see**. In fact, in many places throughout this reading notebook, you will be asked to write your own code to experiment with a concept that was just covered. This is a form of "active reading" and the idea behind it is that we really learn by **doing**. 
<br />

### Change everything
But don't feel limited to only change code when I prompt you. This notebook is your learning environment and your playground. I encourage you to try changing and running all the code throughout the notebook and even to **add your own notes and new code blocks**. Adding comments to code to explain what you are testing, experimenting with or trying to do is really helpful to understand what you were thinking when you revisit it later. 
<br />

### Make this notebook your own
Make this notebook your own. Write your questions and thoughts. At the end of every reading notebook, I will ask the same set of questions to try to elicit your questions, reaction and feedback. When we review the reading notebook in class, I encourage you to share these.
</details>

## Learning Objectives

By the end of this lecture, you will be able to:
- Use Pydantic for data validation and structured data handling
- Work efficiently with large datasets using modern file formats (Parquet, Feather)
- Handle data that doesn't fit in memory using chunking and generators
- Leverage PyArrow backend for improved pandas performance
- Optimize memory usage for data science workflows

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🧩
  </span>

  This lecture focuses on practical skills for working with data at scale. Memory management and file format choices might seem like "boring" infrastructure topics, but they're often the difference between a successful data science project and one that grinds to a halt. **Master these fundamentals and you'll be prepared for real-world data challenges!**
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

# 5.0 Code Preface

In [1]:
from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

# 5.1 Pydantic: Data Validation That Actually Works

<img alt="Pydantic validation is like the thorough security in Total Recall" src="../images/L03_security_total_recall.png" width="900" style="display:block;">
<font size=2>Arnie goes through detailed security scanning in <i>Total Recall (1990)</i></font>

Remember how we said type hints aren't enforced at runtime? That's usually fine, but sometimes you *need* validation—especially when dealing with external data like JSON files, API responses, or user input. You want your data to go through a rigorous check.

Enter **Pydantic**: a library that uses type hints to validate data at runtime. It's become incredibly popular in the Python ecosystem, especially for building APIs and handling configuration.

## First: What Are Dataclasses?

A data class is a class specifically designed to mainly hold data. Before we dive into Pydantic, let's quickly look at building a class for data using an ordinary class, then using Python's built-in `dataclasses`. Suppose we want to build a data class to hold details of a Movie object, with attributes like `title`, `year`, and `rating`. Here's how we might do it with a regular class:

In [2]:
# Regular class - lots of boilerplate
class MovieOld:
    def __init__(self, title: str, year: int, rating: float):
        self.title = title
        self.year = year
        self.rating = rating
    
    def __repr__(self):
        return f"Movie(title='{self.title}', year={self.year}, rating={self.rating})"

# Create a movie of type MovieOld
movie1 = MovieOld("Aliens", 1986, 8.4)
print(movie1)

Movie(title='Aliens', year=1986, rating=8.4)


A significant amount of boilerplate code is required to create a simple class that just holds data. But with Python's `dataclasses`, we can achieve the same functionality with much less code:

In [3]:
from dataclasses import dataclass

# Dataclass - much cleaner!
@dataclass
class Movie:
    title: str
    year: int
    rating: float

# Both work the same way
movie2 = Movie("Aliens", 1986, 8.4)
print(movie2)

Movie(title='Aliens', year=1986, rating=8.4)


<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  The <code>@dataclass</code> is an example of a **decorator**. A decorator is a function that wraps another function or class to modify its behavior. When you write <code>@dataclass</code> above a class definition, Python passes your class through the <code>dataclass()</code> function, which automatically generates <code>__init__</code>, <code>__repr__</code>, and other boilerplate methods.

  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

Dataclasses are great, but they have a limitation: **they don't validate data**. Watch what happens:

In [4]:
# Dataclass accepts wrong types without complaint!
bad_movie = Movie(
    title=12345,       # Should be str, but no error!
    year="nineteen eighty-six",  # Should be int, but no error!
    rating="excellent"  # Should be float, but no error!
)
print(bad_movie)
print(f"Year type: {type(bad_movie.year)}")  # It's a string, not an int!

Movie(title=12345, year='nineteen eighty-six', rating='excellent')
Year type: <class 'str'>


## Enter Pydantic: BaseModel

<img alt="pydantic" src="../images/L03_pydantic.png" width="500" style="display:block;">

Pydantic's `BaseModel` is like a dataclass that actually enforces types. It will either **convert data to the right type or *raise an error* if it can't**:

In [ ]:
pip install pydantic

In [5]:
from pydantic import BaseModel

class MovieModel(BaseModel):
    title: str
    year: int
    rating: float

# This works perfectly
good_movie = MovieModel(title="Predator", year=1987, rating=7.8)
print(good_movie)

title='Predator' year=1987 rating=7.8


Now suppose we create a movie object and pass the year as a string isntead of an integer:

In [6]:
# Pydantic will even convert types when possible
converted_movie = MovieModel(title="Lethal Weapon", year="1987", rating="7.6")
print(converted_movie)
print(f"Year type: {type(converted_movie.year)}")  # Converted the string to an int automatically

title='Lethal Weapon' year=1987 rating=7.6
Year type: <class 'int'>


When it is not possible to autoconvert types, rather than silently permit it (like an ordinary Python class or dataclass would), Pydantic raises a `ValidationError` with detailed information about what went wrong:

In [7]:
# But when conversion isn't possible, you get a clear error
try:
    bad_movie = MovieModel(
        title="Unknown",
        year="not a number",  # Can't convert this to int!
        rating=5.0
    )
except Exception as e:
    print(f"Validation error: {e}")

Validation error: 1 validation error for MovieModel
year
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not a number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing


## Field Constraints: More Than Just Types

Pydantic lets you add constraints beyond simple types. Use the `Field` function to specify rules. Here's an example of a Pydantic model for an ML Experiment with some sophisticated type restrictions and validation:

In [8]:
from pydantic import BaseModel, Field
from typing import Optional

class MLExperiment(BaseModel):
    """Configuration for a machine learning experiment."""
    
    name: str = Field(..., min_length=1, max_length=100, description="Experiment name")
    model_type: str = Field(..., pattern="^(random_forest|gradient_boost|neural_net)$")
    learning_rate: float = Field(default=0.01, gt=0, le=1.0)
    n_estimators: int = Field(default=100, ge=10, le=1000)
    random_seed: Optional[int] = Field(default=None, ge=0)
    
# Valid experiment
experiment = MLExperiment(
    name="Customer Churn Prediction",
    model_type="random_forest",
    learning_rate=0.1,
    n_estimators=200
)
print(experiment)

name='Customer Churn Prediction' model_type='random_forest' learning_rate=0.1 n_estimators=200 random_seed=None


Let's break down those `Field` constraints:
- `...` (Ellipsis) - Required field (no default)
- `min_length`, `max_length` - String length limits
- `pattern` - Regular expression the string must match
- `gt`, `ge` - Greater than, greater than or equal
- `lt`, `le` - Less than, less than or equal
- `default` - Default value if not provided
- `description` - Documentation for the field

Just like before, if you try to create an object that violates any of these constraints, Pydantic will raise a `ValidationError` with details about which fields failed and why:

In [9]:
# Invalid experiment - violates constraints
try:
    bad_experiment = MLExperiment(
        name="",  # Too short!
        model_type="decision_tree",  # Not in allowed list!
        learning_rate=5.0,  # Greater than 1.0!
        n_estimators=5  # Less than 10!
    )
except Exception as e:
    print("Validation errors:")
    print(e)

Validation errors:
4 validation errors for MLExperiment
name
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_short
model_type
  String should match pattern '^(random_forest|gradient_boost|neural_net)$' [type=string_pattern_mismatch, input_value='decision_tree', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_pattern_mismatch
learning_rate
  Input should be less than or equal to 1 [type=less_than_equal, input_value=5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
n_estimators
  Input should be greater than or equal to 10 [type=greater_than_equal, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal


<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  You might not have seen <code>...</code> (Ellipsis) in Python before. These aren't just three dots, but actually a special <code>Ellipsis</code> type object that has a few uses in python, often as a literal placeholder (where code will eventually go), as a signal of arbitrary length, to indicate "everything else" when index slicing in NumPy, and many other uses. Here it is used as a "sentinel" to indicate that a Pydantic Field is required and not optional.
  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

## Parsing JSON and Dictionaries

<img alt="Mrs. Voorhees' sweater" src="../images/L05_parsing_jason.png" width="900" style="display:block;">
<font size=2>Mrs. Voorhees' sweater helped Alice parse Jason in <i>Friday the 13th part 2 (1981)</i></font>

One of Pydantic's killer features is parsing data from JSON (and dictionaries). This is incredibly useful when loading configuration files or receiving API responses:

In [10]:
import json

# Imagine this string came from a JSON config file 
config_json = '''
{
    "name": "Revenue Forecasting",
    "model_type": "gradient_boost",
    "learning_rate": 0.05,
    "n_estimators": 500,
    "random_seed": 42
}
'''

# Parse JSON string to a dict, then validate with Pydantic
config_dict = json.loads(config_json) # json.loads parses a JSON string to a Python dict
experiment = MLExperiment(**config_dict)  # Create a pydantic object of type MLExperiment; note: ** unpacks the dictionary to keyword arguments
print(experiment)

name='Revenue Forecasting' model_type='gradient_boost' learning_rate=0.05 n_estimators=500 random_seed=42


<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  The <code>**</code> operator is used in <code>SomeClass(**someDict)</code> to unpack a dictionary into keyword args. It is equivalent to <code>SomeClass(key1=val1, key2=val2, ..., keyn=valn)</code> for all key/vals in the dictionary. You can also use it in function calls. Similarly, there is <code>*someList</code> used to expand positional arguments. A common usage of both is to specify arbitrary positional and keyword arguments in a method/function definition: <code>def SomeFunc(*args,**kwargs)</code>.
  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

In [11]:
# Or we can use the static model_validate_json method directly (Pydantic v2) to instantiate directly from the JSON string:
experiment2 = MLExperiment.model_validate_json(config_json)
print(f"\nSame result: {experiment2}")


Same result: name='Revenue Forecasting' model_type='gradient_boost' learning_rate=0.05 n_estimators=500 random_seed=42


The above use case is common in data science projects where you often have configuration files in JSON format that specify parameters for your experiments. Pydantic allows you to easily load these configurations and validate them with minimal code.

## Converting Back to Dict/JSON

You can also convert Pydantic models back to dictionaries or JSON:

In [12]:
# Convert Pydantic model object to a dictionary
experiment_dict = experiment.model_dump()
print("As dictionary:")
print(experiment_dict)

As dictionary:
{'name': 'Revenue Forecasting', 'model_type': 'gradient_boost', 'learning_rate': 0.05, 'n_estimators': 500, 'random_seed': 42}


In [13]:
# Convert Pydantic model object to a JSON string
experiment_json = experiment.model_dump_json(indent=2)
print("\nAs JSON:")
print(experiment_json)


As JSON:
{
  "name": "Revenue Forecasting",
  "model_type": "gradient_boost",
  "learning_rate": 0.05,
  "n_estimators": 500,
  "random_seed": 42
}


## Nested Models

<img alt="Barbarella in Pygar's nest" src="../images/L05_nest_barbarella.png" width="900" style="display:block;">
<font size=2>Barbarella nests with Pygar in <i>Barbarella (1968)</i></font>

Honestly, the combination of JSON parsing/validation and the ability to create nested models for data classes is probably Pydantic's biggest selling point. This can easily be seen in ML workflows (example below), but is super useful in LLM data extraction workflows, where LLMs get unstructured data as input and must return output in a highly structured schema. This often involves nested data structures. Imagine extracting data from a resume. You might have a top-level `Resume` model that contains nested models for `Education`, `Experience`, `Skills`, etc. 

For now, here's a simple example of using nested models for an ML workflow: 

In [14]:
from typing import Optional
from pydantic import BaseModel, Field

# We'll start with two simple models, a DatasetConfig and a ModelConfig
class DatasetConfig(BaseModel):
    """Configuration for a dataset."""
    path: str
    target_column: str
    test_size: float = Field(default=0.2, gt=0, lt=1)
    
class ModelConfig(BaseModel):
    """Configuration for a model."""
    name: str
    n_estimators: int = Field(default=100, ge=1)
    max_depth: Optional[int] = None


# Now we'll create an ExperimentConfig that nests our DatasetConfig and ModelConfig defined above as part of the configuration
class ExperimentConfig(BaseModel):
    """Full experiment configuration with nested models."""
    experiment_name: str
    dataset: DatasetConfig  # Nested model!
    model: ModelConfig      # Another nested model!
    output_dir: str = "./results"


Now that we've defined our nested models, here's how we can create an instance of an object of type `MLExperiment`:

In [15]:

# Create a pydantic object with nested structure
config = ExperimentConfig(
    experiment_name="Churn Analysis Q4",
    dataset=DatasetConfig(
        path="data/customers.csv",
        target_column="churned",
        test_size=0.25
    ),
    model=ModelConfig(
        name="RandomForest",
        n_estimators=200,
        max_depth=10
    )
)

print(f"Experiment: {config.experiment_name}")
print(f"Dataset path: {config.dataset.path}")
print(f"Model: {config.model.name} with {config.model.n_estimators} estimators")

Experiment: Churn Analysis Q4
Dataset path: data/customers.csv
Model: RandomForest with 200 estimators


Or (and this is the real magic) we can create an `MLExperiment` object directly from a JSON that matches the structure of our models:

In [ ]:
# Parse the same structure from JSON
config_json = '''
{
    "experiment_name": "Sales Prediction",
    "dataset": {
        "path": "data/sales.parquet",
        "target_column": "revenue",
        "test_size": 0.3
    },
    "model": {
        "name": "GradientBoosting",
        "n_estimators": 500
    }
}
'''

config_from_json = ExperimentConfig.model_validate_json(
    Path(r"C:\Users\Lexi\Desktop\Spring 2026\BUS 675 Adv Bus ML\Code\bus675.json").read_text()
)

print(config_from_json)

## Use Cases in ML and Analytics:

Pydantic is especially valuable for:

1. **Configuration Files**: Define your experiment configs as Pydantic models. Load them from YAML/JSON and get automatic validation.

2. **API Responses**: When working with APIs (like calling an ML service), validate the response structure.

3. **Data Schemas**: Define what your input data should look like before processing.

4. **Pipeline Parameters**: Pass validated configuration between pipeline stages.

As you can imagine, Pydantic validation is useful for cases when we need to ensure that nothing gets through that doesn't match our expected structure.

<!-- Start Exercise 5.1 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Create a Pydantic Model </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Create a Pydantic model called <code>VillainApplication</code> for the League of Questionable Morals registration system. It should have:
<ul>
<li><code>villain_name</code>: required string, 3-40 characters</li>
<li><code>real_name</code>: optional string (secret identity protection)</li>
<li><code>evil_laugh_style</code>: one of ["maniacal", "sinister", "awkward_chuckle", "silent_stare"]</li>
<li><code>nemesis_name</code>: optional string (not everyone has one yet)</li>
<li><code>weakness</code>: required string (for insurance purposes)</li>
<li><code>world_domination_attempts</code>: integer >= 0, default 0</li>
<li><code>lair_location</code>: one of ["volcano_island", "abandoned_warehouse", "parents_basement", "moon_base", "underwater_dome", "definitely_not_that_building"]</li>
</ul>

Then create an instance and test that validation works.</div>

In [23]:
from typing import Optional
from pydantic import BaseModel, Field

class VillainApplication(BaseModel):
    villain_name: str = Field(..., min_length=3, max_length=40, description="villain name")
    real_name: Optional[str] = None
    evil_laugh_style: str = Field(..., pattern=r"^(maniacal|sinister|awkward_chuckle|silent_stare)$")
    nemesis_name: Optional[str] = None
    weakness: str
    world_domination_attempts: int = Field(default=0, ge=0)
    lair_location: str = Field(..., pattern=r"^(volcano_island|abandoned_warehouse|parents_basement|moon_base)$")

villain = VillainApplication(
    villain_name="Dr. Chaos",
    weakness="kittens",
    evil_laugh_style="maniacal",
    lair_location="volcano_island",
    world_domination_attempts=3
)
print(villain)


villain_name='Dr. Chaos' real_name=None evil_laugh_style='maniacal' nemesis_name=None weakness='kittens' world_domination_attempts=3 lair_location='volcano_island'


<hr/>
<!-- End Exercise 4.5 -->

# 5.2 Efficient File Formats for Large Data

<img alt="Different file formats are like different storage containers - some are more efficient than others" src="../images/L05_parquet_feather_csv.png" width="800" style="display:block;">
<font size=2>Choosing the right file format is imperative, especially when handling large data.</font>

When you're working with small datasets, CSV files are convenient and easy to use. But as your data grows into millions of rows, CSVs become a bottleneck:
- **Slow to read**: CSVs are text-based and require parsing every field
- **No type information**: Every time you load, pandas has to guess column types
- **Large file sizes**: No built-in compression
- **Full file reads**: You can't easily read just the columns you need

Modern file formats like **Parquet** and **Feather** solve these problems.

Before we dive into the details, we'll need to create some artificial data. Run this code block to create a large dataframe in memory:

In [2]:
import numpy as np
import pandas as pd

# First, let's create a sample DataFrame to work with
np.random.seed(42)

n_rows = 1_000_000

df = pd.DataFrame({
    'transaction_id': range(n_rows),
    'date': pd.date_range('2023-01-01', periods=n_rows, freq='min'),
    'customer_id': np.random.randint(1, 10000, n_rows),
    'product_id': np.random.randint(1, 1000, n_rows),
    'amount': np.random.uniform(10, 500, n_rows).round(2),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Books', 'Home'], n_rows),
    'status': np.random.choice(['completed', 'pending', 'refunded'], n_rows, p=[0.85, 0.10, 0.05])
})

print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()

DataFrame shape: (1000000, 7)
Memory usage: 138.42 MB


,transaction_id,date,customer_id,product_id,amount,category,status
0,0,2023-01-01 00:00:00,7271,444,417.95,Food,pending
1,1,2023-01-01 00:01:00,861,683,172.65,Food,completed
2,2,2023-01-01 00:02:00,5391,54,244.39,Electronics,completed
3,3,2023-01-01 00:03:00,5192,375,318.48,Home,completed
4,4,2023-01-01 00:04:00,5735,932,49.86,Food,completed


## CSV: The old standby. Universally supported but inefficient for large data.

The `CSV` (Comma-Separated Values) format is the most widely used file format for tabular data. It's human-readable and supported by virtually every tool, but it has significant limitations when it comes to performance and efficiency, especially with large datasets.

Let's save our dataframe to CSV and see how it performs:

In [3]:
import time
from pathlib import Path


# Create a data directory if it doesn't exist
data_dir = Path("../data/lecture_05")
data_dir.mkdir(parents=True, exist_ok=True)

# Save as CSV
csv_path = data_dir / "transactions.csv"
start = time.time()
df.to_csv(csv_path, index=False)
csv_write_time = time.time() - start
csv_size = csv_path.stat().st_size / 1024**2

print(f"CSV write time: {csv_write_time:.2f} seconds")
print(f"CSV file size: {csv_size:.2f} MB")

CSV write time: 4.94 seconds
CSV file size: 57.73 MB


## Parquet: The Data Science Standard

**Parquet** is a columnar storage format that has become the standard for data engineering and large-scale data science. It was created by Apache and is used by tools like Spark, Databricks, and AWS Athena.

Key benefits:
- **Columnar storage**: Data is stored by column, not row, making it fast to read specific columns
- **Built-in compression**: Files are typically 5-10x smaller than CSVs
- **Type preservation**: Column types are stored in metadata, no guessing needed
- **Schema evolution**: You can add columns without breaking compatibility

Let's see how Parquet performs for writing our dataframe to disk, compared to CSV:

In [ ]:
%pip install pyarrow


In [ ]:
%pip install fastparquet

In [4]:
# Save as Parquet
parquet_path = data_dir / "transactions.parquet"
start = time.time()
df.to_parquet(parquet_path, index=False)
parquet_write_time = time.time() - start
parquet_size = parquet_path.stat().st_size / 1024**2

print(f"Parquet write time: {parquet_write_time:.2f} seconds")
print(f"Parquet file size: {parquet_size:.2f} MB")
print(f"\nParquet is {csv_size / parquet_size:.1f}x smaller than CSV!")

Parquet write time: 0.74 seconds
Parquet file size: 16.66 MB

Parquet is 3.5x smaller than CSV!


## Feather: Optimized for DataFrames

**Feather** is another columnar format, designed specifically for fast DataFrame serialization. It's particularly good for saving intermediate results or caching data.

Key benefits:
- **Extremely fast read/write**: Optimized for pandas/Arrow DataFrames
- **Preserves data types perfectly**: Including categories, datetimes, etc.
- **Simple to use**: No configuration needed

Trade-offs vs Parquet:
- Feather files are typically slightly larger
- Less widely supported by other tools
- Best for within-Python workflows or data that might be read into R (for statistical modelling)

In [5]:
# Save as Feather
feather_path = data_dir / "transactions.feather"
start = time.time()
df.to_feather(feather_path)
feather_write_time = time.time() - start
feather_size = feather_path.stat().st_size / 1024**2

print(f"Feather write time: {feather_write_time:.2f} seconds")
print(f"Feather file size: {feather_size:.2f} MB")

Feather write time: 0.35 seconds
Feather file size: 31.97 MB


## Reading Performance Comparison

Now let's compare how fast we can read from each format:

In [6]:
# Read CSV
start = time.time()
df_csv = pd.read_csv(csv_path)
csv_read_time = time.time() - start
print(f"CSV read time: {csv_read_time:.3f} seconds")

CSV read time: 1.550 seconds


In [7]:
# Read Parquet
start = time.time()
df_parquet = pd.read_parquet(parquet_path)
parquet_read_time = time.time() - start
print(f"Parquet read time: {parquet_read_time:.3f} seconds")

Parquet read time: 0.799 seconds


In [8]:
# Read Feather
start = time.time()
df_feather = pd.read_feather(feather_path)
feather_read_time = time.time() - start
print(f"Feather read time: {feather_read_time:.3f} seconds")

Feather read time: 0.213 seconds


In [9]:

print(f"\nParquet is {csv_read_time / parquet_read_time:.1f}x faster than CSV")
print(f"Feather is {csv_read_time / feather_read_time:.1f}x faster than CSV")


Parquet is 1.9x faster than CSV
Feather is 7.3x faster than CSV


## Reading Selected Columns (Parquet Advantage)

One of Parquet's biggest advantages is **column pruning** - you can read only the columns you need without loading the entire file:

In [10]:
# Read only specific columns from Parquet
start = time.time()
df_subset = pd.read_parquet(parquet_path, columns=['customer_id', 'amount', 'category'])
parquet_subset_time = time.time() - start

print(f"Reading 3 columns from Parquet: {parquet_subset_time:.3f} seconds")
print(f"Shape: {df_subset.shape}")
print(f"\nThis is much faster than reading the full file when you only need a few columns!")

Reading 3 columns from Parquet: 0.123 seconds
Shape: (1000000, 3)

This is much faster than reading the full file when you only need a few columns!


## Summary: Format Comparison

| Feature | CSV | Parquet | Feather |
|---------|-----|---------|---------|
| **File Size** | Largest | Smallest | Medium |
| **Read Speed** | Slowest | Fast | Fastest |
| **Write Speed** | Medium | Medium | Fastest |
| **Column Selection** | No | Yes | Limited |
| **Type Preservation** | No | Yes | Yes |
| **Tool Compatibility** | Universal | Wide | Python- and R-focused |
| **Best For** | Sharing, simple data | Big data, analytics | Caching, intermediate |

<font size=2>\* the above is generally true for large files, but for smaller files, the overhead of parquet and feather can dominate leading to worse performance than csv</font>

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🧠
  </span>

<font color=blue><b>Q</b></font>: Which format should I use?

<font color=blue><b>A</b></font>: Use <b>Parquet</b> or <b>Feather</b> for sharing data in ML/analytics collabs. For cloud analytics tools, Parquet is preferred. <b>Feather</b> if you want pure speed or for trading data between Python and R. Keep <b>CSV</b> to maximize compatibility and sharing broadly.
  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

<!-- Start Exercise 5.2 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: File Format Comparison </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Load a CSV file, convert it to both Parquet and Feather formats, compare file sizes and load times, then read only specific columns from the Parquet file. Use the transactions.csv file we created above.
</div>

In [ ]:
# Exercise: File format comparison
import pandas as pd
from pathlib import Path


# 1. Load the CSV file
# 2. Save as Parquet with compression='snappy' and compression='gzip'
# 3. Compare file sizes
# 4. Read only ['date', 'amount'] columns from each Parquet file

# Your code here
# TO-DO

<hr/>
<!-- End Exercise 5.2 -->

# 5.3 Handling Large Datasets

<img alt="Shel Silverstein whale" src="../images/L05_whale_poem.png" width="900" style="display:block;">
<font size=2>How do you eat a whale? One bite at a time. Same with big data.    <i> Where the Sidewalk Ends (1974)</i> by Shel Silverstein </font>

What happens when your dataset is too large to fit in memory? You have several options:
1. **Chunked reading**: Process the data in pieces
2. **Generators with yield**: Lazy evaluation for memory efficiency
3. **Type optimization**: Reduce memory footprint of loaded data
4. **Filtering during load**: Only load what you need

## Reading Data in Chunks

The `chunksize` parameter in `pd.read_csv()` returns an iterator that yields DataFrames in chunks:

In [11]:
# Read CSV in chunks of 10,000 rows
chunk_size = 10_000

# This returns an iterator, not a DataFrame!
chunks = pd.read_csv(csv_path, chunksize=chunk_size)

print(f"Type of chunks: {type(chunks)}")
print("Processing chunks...")

total_rows = 0
total_amount = 0

for i, chunk in enumerate(chunks):
    # Process each chunk
    total_rows += len(chunk)
    total_amount += chunk['amount'].sum()
    print(f"  Chunk {i+1}: {len(chunk)} rows, running total amount: ${total_amount:,.2f}")

print(f"\nTotal rows processed: {total_rows}")
print(f"Total amount: ${total_amount:,.2f}")

Type of chunks: <class 'pandas.io.parsers.readers.TextFileReader'>
Processing chunks...
  Chunk 1: 10000 rows, running total amount: $2,551,537.98
  Chunk 2: 10000 rows, running total amount: $5,107,856.41
  Chunk 3: 10000 rows, running total amount: $7,678,931.11
  Chunk 4: 10000 rows, running total amount: $10,222,556.01
  Chunk 5: 10000 rows, running total amount: $12,779,164.84
  Chunk 6: 10000 rows, running total amount: $15,316,089.64
  Chunk 7: 10000 rows, running total amount: $17,848,262.07
  Chunk 8: 10000 rows, running total amount: $20,399,989.79
  Chunk 9: 10000 rows, running total amount: $22,941,032.37
  Chunk 10: 10000 rows, running total amount: $25,485,953.98
  Chunk 11: 10000 rows, running total amount: $28,014,622.79
  Chunk 12: 10000 rows, running total amount: $30,556,467.09
  Chunk 13: 10000 rows, running total amount: $33,102,803.26
  Chunk 14: 10000 rows, running total amount: $35,639,051.16
  Chunk 15: 10000 rows, running total amount: $38,192,545.46
  Chunk 1

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    💡
  </span>

  When using chunked reading, you can only iterate through the chunks <b>once</b>. This is a common trade-off when handling large data files. In such cases, it's important to think carefully about your data processing workflow and compare different approaches with real-world performance benchmarks.
  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

## Generators and `yield`: Memory-Efficient Processing

Python generators are functions that produce a sequence of values lazily - computing each value only when needed. They use the `yield` keyword instead of `return`. Have a look at these two version of the same function for generating the squares of numbers -- one using a list and the other using a generator:

In [12]:
# Regular function - creates entire list in memory
def get_squares_list(n):
    result = []  # notice we create the list in the function and return it (so Python has to keep the whole list in memory)
    for i in range(n):
        result.append(i ** 2)
    return result

# Generator function - yields one value at a time
def get_squares_generator(n):
    for i in range(n):  # Here we don't "save" the list anywhere, we just "yield" one value at a time (whenever they are asked for)
        yield i ** 2


Let's see how they compare from a memory standpoint:

In [13]:

# Compare memory usage
import sys

# List approach - all in memory
squares_list = get_squares_list(1000000)
print(f"List size: {sys.getsizeof(squares_list):,} bytes")

# Generator approach - lazy evaluation
squares_gen = get_squares_generator(1000000)
print(f"Generator size: {sys.getsizeof(squares_gen):,} bytes")

List size: 8,448,728 bytes
Generator size: 200 bytes


Getting values from the generator is a bit strange at first if you're used to regular functions. You can use the `next()` function to get the next value from the generator:

In [14]:
# Using the generator
squares_gen = get_squares_generator(5)

# Each call to next() gets the next value
print(next(squares_gen))  # 0
print(next(squares_gen))  # 1
print(next(squares_gen))  # 4

0
1
4


But this isn't how you typically use generators. More often, you'll iterate through them in a loop:

In [15]:
# We often iterate through generators in a loop
squares_gen = get_squares_generator(5)

for square in squares_gen:
    print(square)  # 9, 16

0
1
4
9
16


You can think of a generator as "pausing" the function at the `yield` statement and resuming from there the next time you call `next()` or iterate. This allows you to produce values one at a time without storing the entire sequence in memory.

## Real-World Generator: Processing Large Files

Here's a practical example - a generator that processes a large file and yields filtered batches:

In [16]:
def process_transactions_in_batches(file_path, batch_size=10_000, min_amount=100):
    """
    Generator that reads a CSV file in chunks and yields filtered batches.
    
    Only yields rows where amount >= min_amount.
    """
    chunks = pd.read_csv(file_path, chunksize=batch_size)
    
    for chunk_num, chunk in enumerate(chunks):
        # Filter for high-value transactions
        filtered = chunk[chunk['amount'] >= min_amount]
        
        if len(filtered) > 0:
            yield {
                'batch_num': chunk_num,
                'data': filtered,
                'n_rows': len(filtered),
                'total_amount': filtered['amount'].sum()
            }

# Use the generator
print("Processing high-value transactions:")
for batch in process_transactions_in_batches(csv_path, batch_size=25_000, min_amount=200):
    print(f"  Batch {batch['batch_num']}: {batch['n_rows']} rows, total ${batch['total_amount']:,.2f}")

Processing high-value transactions:
  Batch 0: 15369 rows, total $5,375,869.08
  Batch 1: 15283 rows, total $5,359,549.19
  Batch 2: 15256 rows, total $5,321,337.93
  Batch 3: 15238 rows, total $5,339,136.50
  Batch 4: 15224 rows, total $5,312,000.37
  Batch 5: 15307 rows, total $5,352,525.70
  Batch 6: 15203 rows, total $5,317,968.44
  Batch 7: 15353 rows, total $5,387,339.15
  Batch 8: 15335 rows, total $5,367,178.35
  Batch 9: 15240 rows, total $5,337,222.03
  Batch 10: 15196 rows, total $5,316,408.29
  Batch 11: 15250 rows, total $5,346,584.63
  Batch 12: 15300 rows, total $5,340,444.71
  Batch 13: 15206 rows, total $5,295,995.91
  Batch 14: 15380 rows, total $5,378,054.01
  Batch 15: 15310 rows, total $5,376,901.29
  Batch 16: 15335 rows, total $5,366,759.44
  Batch 17: 15353 rows, total $5,375,589.01
  Batch 18: 15317 rows, total $5,358,247.90
  Batch 19: 15234 rows, total $5,347,901.54
  Batch 20: 15124 rows, total $5,300,607.30
  Batch 21: 15283 rows, total $5,329,752.87
  Batc

## Memory Optimization: dtype Optimization

Often we work with large data that we extract or read from upstream sources. By default, pandas uses data types that are more general and can consume more memory than necessary. For example, it might use `int64` for integer columns even if the values fit in `int8`, or `object` for string columns that could be categorical.

One of the simplest ways to reduce memory usage is to use more appropriate data types:

In [17]:
# Load data and check memory usage
df = pd.read_csv(csv_path)

print("Original memory usage:")
print(df.memory_usage(deep=True))
print(f"\nTotal: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Original memory usage:
Index                  132
transaction_id     8000000
date              68000000
customer_id        8000000
product_id         8000000
amount             8000000
category          55398343
status            57749826
dtype: int64

Total: 203.27 MB


In [18]:
# Optimize dtypes
df_optimized = df.copy()

# Convert object columns with few unique values to category
for col in ['category', 'status']:
    df_optimized[col] = df_optimized[col].astype('category')

# Convert integers to smaller types if possible
df_optimized['customer_id'] = df_optimized['customer_id'].astype('int32')
df_optimized['product_id'] = df_optimized['product_id'].astype('int32')
df_optimized['transaction_id'] = df_optimized['transaction_id'].astype('int32')

# Convert amount to float32 (loses some precision but saves memory)
df_optimized['amount'] = df_optimized['amount'].astype('float32')

print("Optimized memory usage:")
print(df_optimized.memory_usage(deep=True))
print(f"\nTotal: {df_optimized.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

original_mem = df.memory_usage(deep=True).sum()
optimized_mem = df_optimized.memory_usage(deep=True).sum()
print(f"\nMemory reduction: {(1 - optimized_mem/original_mem)*100:.1f}%")

Optimized memory usage:
Index                  132
transaction_id     4000000
date              68000000
customer_id        4000000
product_id         4000000
amount             4000000
category           1000449
status             1000279
dtype: int64

Total: 82.02 MB

Memory reduction: 59.7%


These savings may not seem like a big deal, but when you have data on the scale of dozens of Terabytes, these optimizations can make a huge difference in performance and resource usage.

## Filtering During Load

Why load data you don't need? Use the `usecols` parameter to read only specific columns:

In [19]:
# Load only the columns we need
columns_needed = ['date', 'customer_id', 'amount', 'category']

start = time.time()
df_full = pd.read_csv(csv_path)
full_time = time.time() - start

start = time.time()
df_subset = pd.read_csv(csv_path, usecols=columns_needed)
subset_time = time.time() - start

print(f"Full load: {full_time:.3f}s, shape {df_full.shape}")
print(f"Subset load: {subset_time:.3f}s, shape {df_subset.shape}")
print(f"\nSubset is {full_time/subset_time:.1f}x faster")

Full load: 1.527s, shape (1000000, 7)
Subset load: 1.268s, shape (1000000, 4)

Subset is 1.2x faster


<!-- Start Exercise 5.3 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred"> Exercise: Write a Batch Generator </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Write a generator function that:
<ol>
<li>Reads a CSV file in chunks of a specified size</li>
<li>Filters each chunk to keep only rows where <code>category == 'Electronics'</code></li>
<li>Yields the filtered chunk along with some summary statistics</li>
</ol>
Then use the generator to process the transactions file and calculate the total Electronics sales.
</div>

In [ ]:
import pandas as pd
csv_path = "../data/lecture_05/transactions.csv" # Make sure you ran the code above to create this file

def electronics_batch_generator(file_path, batch_size=20_000):
    """
    Generator that yields batches of electronics transactions.
    
    Each yield should include:
    - The filtered DataFrame
    - The number of rows
    - The sum of amounts in this batch
    """
    # Your code here
    pass

# Test your generator
# total_electronics_sales = 0
# for batch in electronics_batch_generator(csv_path):
#     total_electronics_sales += batch['total_amount']
# print(f"Total Electronics sales: ${total_electronics_sales:,.2f}")

# TO-DO

<hr/>
<!-- End Exercise 5.3 -->

# 5.4 Pandas with PyArrow Backend

<img alt="PyArrow supercharges pandas like a turbocharger on an engine" src="../images/L05_pyarrow.png" width="500" style="display:block;">
<font size=2>PyArrow is like adding a turbocharger to your pandas engine</font>

Starting with pandas 2.0, you can use **PyArrow** as a backend for DataFrames. PyArrow is the Python implementation of Apache Arrow, a high-performance columnar memory format.

Benefits of PyArrow backend:
- **Faster string operations**: Significant speedup for text processing
- **Better null handling**: Native null support (not NaN)
- **Memory efficiency**: Better memory usage for string-heavy data
- **Interoperability**: Seamless conversion to/from Parquet, Feather

In [ ]:
# Check pandas version
print(f"Pandas version: {pd.__version__}")

# Load with default (numpy) backend
df_numpy = pd.read_csv(csv_path)
print(f"\nDefault backend dtypes:")
print(df_numpy.dtypes)

In [ ]:
# Load with PyArrow backend
df_arrow = pd.read_csv(csv_path, dtype_backend='pyarrow')
print(f"PyArrow backend dtypes:")
print(df_arrow.dtypes)

Notice the difference in dtypes:
- Integers become `int64[pyarrow]` instead of `int64`
- Strings become `string[pyarrow]` instead of `object`
- These Arrow types are more efficient and feature-rich

In [ ]:
# Compare memory usage
numpy_mem = df_numpy.memory_usage(deep=True).sum()
arrow_mem = df_arrow.memory_usage(deep=True).sum()

print(f"NumPy backend memory: {numpy_mem / 1024**2:.2f} MB")
print(f"PyArrow backend memory: {arrow_mem / 1024**2:.2f} MB")
print(f"Difference: {(numpy_mem - arrow_mem) / 1024**2:.2f} MB ({(1-arrow_mem/numpy_mem)*100:.1f}% reduction)")

## String Operations: Where PyArrow Shines

PyArrow's biggest advantage is with string operations. Let's compare:

In [ ]:
# String operation benchmark
import time

# Create DataFrames with string data for benchmarking
n = 100_000

# NumPy backend
df_numpy = pd.DataFrame({
    'text': [f'Category_{i % 100}' for i in range(n)] # generate 1000 rows in each category from 0 to 99
})

# PyArrow backend
df_arrow = df_numpy.copy()
df_arrow['text'] = df_arrow['text'].astype('string[pyarrow]')

# Benchmark: String contains
start = time.time()
result_numpy = df_numpy['text'].str.contains('Category_5')
numpy_time = time.time() - start

start = time.time()
result_arrow = df_arrow['text'].str.contains('Category_5')
arrow_time = time.time() - start

print(f"str.contains() benchmark:")
print(f"  NumPy: {numpy_time:.4f}s")
print(f"  PyArrow: {arrow_time:.4f}s")
print(f"  PyArrow is {numpy_time/arrow_time:.1f}x faster")

In [ ]:
# Benchmark: String replacement
start = time.time()
result_numpy = df_numpy['text'].str.replace('Category', 'Cat')
numpy_time = time.time() - start

start = time.time()
result_arrow = df_arrow['text'].str.replace('Category', 'Cat')
arrow_time = time.time() - start

print(f"\nstr.replace() benchmark:")
print(f"  NumPy: {numpy_time:.4f}s")
print(f"  PyArrow: {arrow_time:.4f}s")
print(f"  PyArrow is {numpy_time/arrow_time:.1f}x faster")

## Better Null Handling

PyArrow has native support for null values, which is cleaner than NumPy's NaN approach. Numpy has  `NaN` but only for floating point dtypes, so it will upconvert to accomodate them:

In [ ]:
# Create data with missing values
data = {'value': [1, 2, None, 4, None, 6]}

# NumPy backend - uses NaN (not a number) for missing integers
df_numpy = pd.DataFrame(data)
print("NumPy backend:")
print(df_numpy)
print(f"dtype: {df_numpy['value'].dtype}")  # Converts to float64!

In [ ]:
# PyArrow backend - native null support
df_arrow = pd.DataFrame(data).convert_dtypes(dtype_backend='pyarrow')
print("\nPyArrow backend:")
print(df_arrow)
print(f"dtype: {df_arrow['value'].dtype}")  # Stays integer with nulls!

## When to Use PyArrow Backend

**Use PyArrow when:**
- Working with string-heavy data
- You need efficient null handling for integers
- Loading Parquet/Feather files (natural fit)
- Processing large datasets where memory matters

**Stick with NumPy when:**
- Using older pandas versions (< 2.0)
- Working with libraries that expect NumPy types
- Doing heavy numerical computations (NumPy is still king)
- Code compatibility is important

<div class="callout" style="
  width: 80%;
  background: rgba(127,127,127,0.15);
  border: 1px solid rgba(127,127,127,0.3);
  padding: 10px 30px;
  margin: 20px;
  border-radius: 6px;
  text-align: justify;
  text-align-last: left;
  font-size: 11pt;
">
  <span style="
    font-size: 60pt;
    line-height: 1;
    float: left;
    margin: 0px 0px 0px 0;
  ">
    🧠
  </span>

PyArrow is becoming the future of pandas. As of pandas 2.0, the recommendation is to start experimenting with PyArrow-backed DataFrames. Future versions of pandas may make PyArrow the default.
  
  <!-- clearfix -->
  <div style="clear: both;"></div>
</div>

<!-- Start Exercise 5.4 -->
<hr/>
<img src="../images/stop_right_margin.png" align="left">

<font size=3 color="darkred">  Exercise: Memory Optimization Challenge </font>
<div class="inclass_exercise_body" style="padding-left: 130px; width: 85%; text-align: justify;text-align-last: left;">
Take a DataFrame with object dtypes and optimize memory by:
<ol>
<li>Converting string columns with few unique values to <code>category</code></li>
<li>Using smaller integer types where possible (<code>int32</code> vs <code>int64</code>)</li>
<li>Converting to PyArrow backend</li>
</ol>
Measure the memory reduction at each step.
</div>

In [ ]:
# Load the transactions data
import pandas as pd
csv_path = "../data/lecture_05/transactions.csv" # Make sure you ran the code above to create this file
df = pd.read_csv(csv_path)


# Step 1: Check original memory
# Your code here

# Step 2: Convert category and status to 'category' dtype
# Your code here

# Step 3: Convert integer columns to int32
# Your code here

# Step 4: Convert to PyArrow backend
# Your code here

# Print final comparison
# final_mem = df.memory_usage(deep=True).sum()
# print(f"Final memory: {final_mem / 1024**2:.2f} MB")
# print(f"Total reduction: {(1 - final_mem/original_mem)*100:.1f}%")

<hr/>
<!-- End Exercise 5.4 -->

# Summary

In this lecture, we covered essential techniques for working with data at scale:

## Pydantic Data Validation
- `BaseModel` for creating validated data structures
- `Field` for adding constraints (min/max, patterns, etc.)
- Parsing JSON/dict data into validated objects
- Nested models for complex configuration structures

## Efficient File Formats
- **Parquet**: Columnar format, great compression, read specific columns
- **Feather**: Fastest read/write for Python workflows
- **CSV**: Universal but slow and large

## Handling Large Datasets
- **Chunked reading**: Process data in pieces with `chunksize`
- **Generators with yield**: Memory-efficient lazy evaluation
- **dtype optimization**: Use appropriate types (category, int32, etc.)
- **Selective loading**: Read only columns you need with `usecols`

## PyArrow Backend
- Enable with `dtype_backend='pyarrow'`
- Faster string operations
- Better null handling (native nulls, not NaN)
- Memory efficiency for string-heavy data

These techniques are essential for real-world data science where datasets often exceed available memory and performance matters.

# Your Turn: Questions, Reactions, and Feedback

As always, I'd like to hear your thoughts on this material:

1. What concepts were most new or surprising to you?

2. What would you like to see explained in more depth?

3. Any questions that came up as you worked through the notebook?

4. What connections do you see between these techniques and the data work you've done before?

**Write your thoughts below:**

*[Your reflections here]*